In [1]:
import pandas as pd

In [2]:
training_data = pd.read_csv('data/v0.1/flextrack-2025-training-data-v0.1.csv', parse_dates=['Timestamp_Local'])
test_data = pd.read_csv('data/v0.1/flextrack-2025-public-test-data-v0.1.csv', parse_dates=['Timestamp_Local'])
random_preds = pd.read_csv('data/v0.1/flextrack-2025-random-prediction-data-v0.1.csv')

In [3]:
time_col = "Timestamp_Local"
site_col = "Site"
index_cols = [site_col, time_col]
features = ["Dry_Bulb_Temperature_C", "Global_Horizontal_Radiation_W/m2", "Building_Power_kW"]
target_col_classification = "Demand_Response_Flag"
target_cols = [target_col_classification]
# target_cols = ["Demand_Response_Flag", "Demand_Response_Capacity_kW"]

### Create Data Splits

In [4]:
X_train = training_data[index_cols + features].copy()
y_train = training_data[index_cols + target_cols].copy()

### Create lags and rolling features

In [5]:
time_offsets = {"1h": 4, "1d": 24 * 4, "7d": 24 * 4 * 7, "m": 24 * 4 * 30, "y": 24 * 4 * 365}

In [6]:
stats = ['mean', 'std', 'min', 'max']
lags = []
for offset in ["1h","1d","7d"]:
    shift = training_data.set_index('Timestamp_Local').shift(freq=offset).reset_index().set_index(['Site',"Timestamp_Local"])
    shift.rename(columns=lambda x: f"{x}_shift_{offset}", inplace=True)
    window_stats = []
    for stat in stats:
        window_stat = training_data.groupby('Site').apply(
            lambda x: x.set_index('Timestamp_Local').rolling(offset, min_periods=time_offsets[offset]).agg(stat), include_groups=False
        ).reset_index().set_index(['Site',"Timestamp_Local"])
        window_stat.rename(columns=lambda x: f"{x}_roll_{offset}_{stat}", inplace=True)
        window_stats.append(window_stat)
    lags.append(pd.concat([shift, *window_stats], axis=1))

In [7]:
def get_lags_features(df, lags):
    stats = ['mean', 'std', 'min', 'max']
    lags_features = []
    for offset in lags:
        shift = df.set_index('Timestamp_Local').shift(freq=offset).reset_index().set_index(['Site',"Timestamp_Local"])
        shift.rename(columns=lambda x: f"{x}_shift_{offset}", inplace=True)
        window_stats = []
        for stat in stats:
            window_stat = df.groupby('Site').apply(
                lambda x: x.set_index('Timestamp_Local').rolling(offset, min_periods=time_offsets[offset]).agg(stat), include_groups=False
            ).reset_index().set_index(['Site',"Timestamp_Local"])
            window_stat.rename(columns=lambda x: f"{x}_roll_{offset}_{stat}", inplace=True)
            window_stats.append(window_stat)
        lags_features.append(pd.concat([shift, *window_stats], axis=1))
    lags_features = pd.concat(lags_features, axis=1)
    return lags_features

In [8]:
lags_features = get_lags_features(X_train, ["1h","1d","7d"])

In [9]:
# X_train = pd.concat([X_train.set_index(index_cols), *lags], axis=1).sort_index().reset_index()
X_train = X_train.set_index(index_cols).join(lags_features).reset_index()

In [10]:
import numpy as np

def add_time_features(df, time_col='Timestamp_Local'):
    df['Minute'] = df[time_col].dt.minute
    df['Hour'] = df[time_col].dt.hour
    df['Day'] = df[time_col].dt.day
    df['Month'] = df[time_col].dt.month
    df['Year'] = df[time_col].dt.year
    df['Day_Of_Week'] = df[time_col].dt.dayofweek
    df['Is_Weekend'] = df['Day_Of_Week'].isin([5, 6]).astype(int)
    
    # Add cyclical features using cosine
    df['Minute_cos'] = np.cos(2 * np.pi * df['Minute'] / 60)
    df['Hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)
    df['Day_cos'] = np.cos(2 * np.pi * df['Day'] / 31)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Day_Of_Week_cos'] = np.cos(2 * np.pi * df['Day_Of_Week'] / 7)
    
    return df

In [11]:
X_train = add_time_features(X_train)

In [12]:
X_train = X_train.set_index(index_cols)

In [13]:
y_train = y_train.set_index(index_cols)
y_train = y_train.loc[X_train.index]

In [14]:
extra_time_festures = ['Hour', 'Month', 'Year', 'Day_Of_Week']

In [15]:
y_train['Demand_Response_Flag'].value_counts(normalize=True)

Demand_Response_Flag
 0    0.970424
-1    0.021518
 1    0.008057
Name: proportion, dtype: float64

### Train CatBoost Classifier
Implement CatBoost training using existing X_train and y_train, evaluate, show feature importances, and save the model.

In [16]:
# Install catboost if not available (uncomment if running in a fresh environment)
# %pip install catboost

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import joblib
import os, json

TARGET = "Demand_Response_Flag"

# Align and drop rows with missing target
train_df = X_train.join(y_train[[TARGET]])
train_df = train_df.dropna(subset=[TARGET])

# Encode multiclass labels (-1,0,1) to (0,1,2) for CatBoost if needed
classes_sorted = sorted(train_df[TARGET].unique().tolist())
class_to_idx = {c:i for i,c in enumerate(classes_sorted)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
train_df['__label__'] = train_df[TARGET].map(class_to_idx)

# Time-based split (last 20% per site)
site_splits = []
for site, grp in train_df.groupby(level=0):
    n = len(grp)
    cut = int(n * 0.8)
    site_splits.append((grp.iloc[:cut], grp.iloc[cut:]))
train_part = pd.concat([p[0] for p in site_splits])
valid_part = pd.concat([p[1] for p in site_splits])

feature_cols = [c for c in train_part.columns if c not in [TARGET, '__label__']]

X_tr = train_part[feature_cols]
y_tr = train_part['__label__']
X_va = valid_part[feature_cols]
y_va = valid_part['__label__']

cat_features = ["Is_Weekend"] if "Is_Weekend" in feature_cols else []

train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
valid_pool = Pool(X_va, y_va, cat_features=cat_features)

label_dist = train_df[TARGET].value_counts(normalize=True)
class_weights = []
max_freq = label_dist.max()
for c in classes_sorted:
    freq = label_dist[c]
    class_weights.append(max_freq / freq)

model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=1200,
    random_seed=321,
    # depth=6,
    # learning_rate=0.05,
    # random_seed=42,
    # l2_leaf_reg=3,
    verbose=100,
    class_weights=class_weights,
    # early_stopping_rounds=200
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

va_preds_idx = model.predict(X_va).astype(int).ravel()
va_probs = model.predict_proba(X_va)
va_preds = [idx_to_class[i] for i in va_preds_idx]
true_labels = [idx_to_class[i] for i in y_va]

print('Classes:', classes_sorted)
print('Accuracy:', accuracy_score(true_labels, va_preds))
print('Macro F1:', f1_score(true_labels, va_preds, average='macro'))
print('Weighted F1:', f1_score(true_labels, va_preds, average='weighted'))
print(classification_report(true_labels, va_preds))
print('Confusion Matrix:\n', confusion_matrix(true_labels, va_preds, labels=classes_sorted))

# Top feature importances
importances = model.get_feature_importance(train_pool)
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importances}) \
    .sort_values('importance', ascending=False).head(40)
print(feat_imp)

os.makedirs('models', exist_ok=True)
model_path = 'models/catboost_classifier_multiclass.cbm'
model.save_model(model_path)
print(f'Model saved to {model_path}')

# Persist with joblib
joblib.dump(model, 'models/catboost_classifier_multiclass.joblib')

# Save class mapping
with open('models/catboost_class_mapping.json', 'w') as f:
    json.dump({'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class}, f, indent=2)
print('Saved class mapping.')


Learning rate set to 0.110046
0:	learn: 0.7366371	test: 0.6441534	best: 0.6441534 (0)	total: 64.3ms	remaining: 1m 17s
100:	learn: 0.9677944	test: 0.1385108	best: 0.6795843 (5)	total: 1.36s	remaining: 14.8s
200:	learn: 0.9911281	test: 0.1459434	best: 0.6795843 (5)	total: 2.67s	remaining: 13.3s
300:	learn: 0.9966753	test: 0.1459434	best: 0.6795843 (5)	total: 3.71s	remaining: 11.1s
400:	learn: 0.9984015	test: 0.1459434	best: 0.6795843 (5)	total: 4.76s	remaining: 9.49s
500:	learn: 0.9990647	test: 0.1459434	best: 0.6795843 (5)	total: 5.99s	remaining: 8.36s
600:	learn: 0.9993080	test: 0.1422468	best: 0.6795843 (5)	total: 7.29s	remaining: 7.26s
700:	learn: 0.9994255	test: 0.1422468	best: 0.6795843 (5)	total: 8.69s	remaining: 6.19s
800:	learn: 0.9995261	test: 0.1422468	best: 0.6795843 (5)	total: 9.97s	remaining: 4.97s
900:	learn: 0.9995890	test: 0.1422414	best: 0.6795843 (5)	total: 10.9s	remaining: 3.62s
1000:	learn: 0.9996352	test: 0.1422414	best: 0.6795843 (5)	total: 11.9s	remaining: 2.36s
1

In [17]:
from imblearn.metrics import geometric_mean_score

geometric_mean_score(true_labels, va_preds)

np.float64(0.662140916637055)

### Test Predictions

In [18]:
X_test = test_data[index_cols + features].copy()

In [19]:
lags_features = get_lags_features(X_test, ["1h","1d","7d"])

In [20]:
X_test = X_test.set_index(index_cols).join(lags_features).reset_index()

In [21]:
X_test = add_time_features(X_test)

In [22]:
from catboost import CatBoostClassifier, Pool
import json, os

# Load model and class mappings
model_path = 'models/catboost_classifier_multiclass.cbm'
class_map_path = 'models/catboost_class_mapping.json'
assert os.path.exists(model_path), 'Model file not found.'
assert os.path.exists(class_map_path), 'Class mapping file not found.'

model = CatBoostClassifier()
model.load_model(model_path)
with open(class_map_path, 'r') as f:
    maps = json.load(f)
idx_to_class = {int(k):v for k,v in maps['idx_to_class'].items()}

# Prepare test feature matrix
X_test_feat = X_test.copy()
X_test_feat = X_test_feat.set_index(index_cols)

# Ensure alignment of features with training model
target_like_cols = ['Demand_Response_Flag', '__label__']
model_features = model.feature_names_  # order expected by model
# Drop any unexpected columns
X_test_feat_aligned = X_test_feat.drop(columns=[c for c in X_test_feat.columns if c not in model_features], errors='ignore')
# Add any missing features as NaN
for f in model_features:
    if f not in X_test_feat_aligned.columns:
        X_test_feat_aligned[f] = float('nan')
# Reorder
X_test_feat_aligned = X_test_feat_aligned[model_features]

cat_features = [i for i,f in enumerate(model_features) if f == 'Is_Weekend']

test_pool = Pool(X_test_feat_aligned, cat_features=cat_features)

pred_idx = model.predict(test_pool).astype(int).ravel()
probs = model.predict_proba(test_pool)

pred_labels = [idx_to_class[i] for i in pred_idx]

prob_cols = {f'proba_class_{idx_to_class[i]}': probs[:, i] for i in range(probs.shape[1])}

pred_df = X_test_feat_aligned.copy()
pred_df['Pred_Class'] = pred_labels
for k,v in prob_cols.items():
    pred_df[k] = v

# Reset index for export
pred_out = pred_df.reset_index()[index_cols + ['Pred_Class'] + list(prob_cols.keys())]

os.makedirs('models', exist_ok=True)
pred_csv_path = 'models/test_predictions.csv'
pred_out.to_csv(pred_csv_path, index=False)
print(f'Saved test predictions to {pred_csv_path}')

pred_out.head()

Saved test predictions to models/test_predictions.csv


,Site,Timestamp_Local,Pred_Class,proba_class_-1,proba_class_0,proba_class_1
0,siteD,2020-01-01 00:00:00,0,0.172555,0.654915,0.172530
1,siteD,2020-01-01 00:15:00,0,0.172555,0.654915,0.172530
2,siteD,2020-01-01 00:30:00,0,0.172555,0.654915,0.172530
3,siteD,2020-01-01 00:45:00,0,0.172698,0.654629,0.172673
4,siteD,2020-01-01 01:00:00,0,0.172698,0.654629,0.172673


In [23]:
pred_out.rename(columns={"Pred_Class":"Demand_Response_Flag"}, inplace=True)

In [24]:
pred_out[["Site","Timestamp_Local","Demand_Response_Flag"]].to_csv('data/submission-v1.0.csv', index=False)